# Create a recipe Programmatically

This notebook is a **tutorial** for new users. It uses the **Poetry** environment from `pyproject.toml`.

**What you are doing:** calling the **Datastore** `RecipeService.Create` with a protobuf `CreateRecipeRequest`. A recipe is:

- **`Name`** — label in the GUI.
- **`Geometries`** — shapes (`marking_path`, `waveguide`, …) with `Shape` (Point, LineString, …) and optional `Attributes`.
- **`Attributes`** — recipe-level metadata, especially **`context.scaling_factor_mm_per_px`**, **`context.width`**, **`context.height`** so execution knows pixel size in mm.
- **`LaserParameters`** — keys such as laser frequencies, speeds, delays (see the full example below).
- **`Relations`** — optional `GeometryRelation` objects (e.g. **`Type="offset"`**) that link a parent waveguide to **parallel** `marking_path` linestrings.

The **second example** below builds one waveguide, two offset marking paths (**`single_pulse`** on one side, **`pulse_train`** on the other), and laser timing parameters typical of recipes tuned in the CLI or GUI.

After connecting to the datastore, an **optional** section (off by default) can call **`HardwareApi`** + **`ReadCameraInfo`** so canvas size matches the laser-head camera; set **`FEMTUM_TRY_HARDWARE_CAMERA=1`** when hardware or a mock is available. Otherwise the notebook uses fixed **4512×4512**.

In [ ]:
# Verify you're using the project virtualenv (e.g. Select Kernel → .venv)
import sys
print(f"Python: {sys.executable}")

## Connect to the Datastore

Point `DatastoreApi` at your datastore endpoint. The storage **`GetHealth`** call uses `v1.storage.HealthRequest` (not the hardware health message).

In [ ]:
from femtum_sdk.api import DatastoreApi
from femtum_sdk.api.contracts.v1.storage.health_pb2 import HealthRequest as StorageHealthRequest


datastore_api = DatastoreApi(host="localhost:5208") # Replace with your datastore endpoint
datastore_api.__enter__()

print("Datastore health:")
print(datastore_api.health.GetHealth(StorageHealthRequest()))

## Optional — Canvas size from the laser-head camera

If a **hardware service** (real device or mock) is reachable, you can set **`context.width`** / **`context.height`** from **`ReadCameraInfo`** after **`ResetROI`**, matching the original notebook flow.

- Set **`FEMTUM_TRY_HARDWARE_CAMERA=1`** when a hardware or mock service is running; it defaults to **off** so datastore-only runs do not probe `localhost:5210`.
- **`FEMTUM_HARDWARE_HOST`** defaults to `localhost:5210` (same port family as datastore `5208`).

When the hardware API is unreachable, the request fails, or the camera reports an invalid size, the next cell falls back to fixed **4512×4512**. **`context.scaling_factor_mm_per_px`** still comes from your calibration setup; adjust **`SCALING_MM_PER_PX`** in the next cell if needed.

In [ ]:
import os

from femtum_sdk.api import HardwareApi
from femtum_sdk.api.contracts.v1.dto.camera_pb2 import CameraInfo
from femtum_sdk.api.contracts.v1.hardware.camera_pb2 import ReadCameraInfoRequest, ResetROIRequest
from femtum_sdk.api.contracts.v1.hardware.health_pb2 import HealthRequest as HardwareHealthRequest

TRY_HARDWARE_CAMERA_FOR_CONTEXT = os.environ.get("FEMTUM_TRY_HARDWARE_CAMERA", "0") == "1"
HARDWARE_HOST = os.environ.get("FEMTUM_HARDWARE_HOST", "localhost:5210")
CONTEXT_WIDTH = os.environ.get("CONTEXT_WIDTH", 4512)
CONTEXT_HEIGHT = os.environ.get("CONTEXT_HEIGHT", 4512)
USED_HARDWARE_CAMERA = os.environ.get("USED_HARDWARE_CAMERA", False)
SCALING_MM_PER_PX = os.environ.get("SCALING_MM_PER_PX", "0.0005736")


if TRY_HARDWARE_CAMERA_FOR_CONTEXT:
    try:
        with HardwareApi(host=HARDWARE_HOST) as hardware_api:
            hw_health = hardware_api.health.GetHealth(HardwareHealthRequest())
            hardware_api.laser_head_camera.ResetROI(ResetROIRequest())
            camera_info: CameraInfo = hardware_api.laser_head_camera.ReadCameraInfo(ReadCameraInfoRequest())
            w = int(camera_info.MaxFrameSize.Width)
            h = int(camera_info.MaxFrameSize.Height)
            if w <= 0 or h <= 0:
                raise ValueError(f"Invalid MaxFrameSize from camera: {w}x{h}")
            CONTEXT_WIDTH, CONTEXT_HEIGHT = w, h
            USED_HARDWARE_CAMERA = True
            print(f"Hardware health: {hw_health}")
            print(f"Canvas from camera (after ResetROI): {CONTEXT_WIDTH} x {CONTEXT_HEIGHT}")
    except Exception as exc:
        print(
            "Hardware camera not used (no service, mock down, or API error). "
            f"{type(exc).__name__}: {exc}"
        )
        print("Next cell will use fixed 4512x4512 unless you set dimensions yourself.")
else:
    
    print(
        "Skipping hardware camera (set FEMTUM_TRY_HARDWARE_CAMERA=1 when hardware/mock is up). "
        "Next cell sets fixed canvas."
    )

## Pixel context (canvas size)

Recipe coordinates are **pixels** on a logical canvas. If the **optional camera** cell succeeded, **`CONTEXT_WIDTH`** / **`CONTEXT_HEIGHT`** are already set from **`MaxFrameSize`**. Otherwise this cell uses **4512×4512** (typical full-frame in these examples).

Tune **`SCALING_MM_PER_PX`** to match your **calibration** (it is not derived from the camera alone).

In [ ]:
cx = CONTEXT_WIDTH / 2
cy = CONTEXT_HEIGHT / 2
_src = "camera" if USED_HARDWARE_CAMERA else "defaults"
print(f"Canvas {_src}: {CONTEXT_WIDTH}x{CONTEXT_HEIGHT} px, center ({cx}, {cy}), scaling {SCALING_MM_PER_PX}")

## Example 1 — Single pulse at the center (minimal recipe)

A good first `Create` call: one **`marking_path`** with a **`Point`** at the canvas center and **`marking_type`** = **`single_pulse`**.

In [ ]:
from femtum_sdk.api.contracts.v1.dto.geometry_pb2 import Attribute, Geometry, Point, Shape
from femtum_sdk.api.contracts.v1.dto.recipe_pb2 import LaserParameter
from femtum_sdk.api.contracts.v1.storage.recipe_pb2 import CreateRecipeRequest

minimal_recipe = datastore_api.recipe.Create(
    CreateRecipeRequest(
        Name="Notebook: single pulse at center",
        Geometries=[
            Geometry(
                Type="marking_path",
                Attributes=[Attribute(Key="marking_type", Value="single_pulse")],
                Shape=Shape(Point=Point(X=cx, Y=cy)),
            )
        ],
        Attributes=[
            Attribute(Key="context.scaling_factor_mm_per_px", Value=SCALING_MM_PER_PX),
            Attribute(Key="context.width", Value=str(CONTEXT_WIDTH)),
            Attribute(Key="context.height", Value=str(CONTEXT_HEIGHT)),
        ],
        LaserParameters=[
            LaserParameter(Key="pulse_train_laser_triggering_frequency_khz", Value="2732", Unit="kHz"),
            LaserParameter(Key="single_pulse_laser_triggering_frequency_khz", Value="2732", Unit="kHz"),
        ],
    )
)
print(f"Created id={minimal_recipe.Id!r} name={minimal_recipe.Name!r}")

## Example 2 — Waveguide + two offset marking paths

1. **Parent** `waveguide` — `LineString` from center to center+(100,100).
2. **`GeometryRelation` with `Type="offset"`** — links the parent to a **child** `marking_path` linestring (`GeometryIds` = `[parent_id, child_id]`).
3. Two parallel marking paths:
   - **`distance_px` = +50** → **`marking_type`** **`single_pulse`** on the child geometry and on the relation.
   - **`distance_px` = -50** → **`pulse_train`** (pulse train along the path) on the child and relation.
4. Child geometries carry **`parent_geometry_id`**, **`marking_path_offset_px`**, and **`marking_type`**.
5. **Geometry `Id`** — **Only required when you use `Relations`:** each `GeometryRelation.GeometryIds` entry must match the `Id` of a geometry in the same `CreateRecipeRequest` (and usually `parent_geometry_id` on children matches the waveguide id). For recipes **without** relations, you can **omit** geometry ids and the datastore will assign UUIDs.
6. **`GeometryRelation.Id`** — **Omit it** unless you have a reason to set it; the server generates a stable id and wires **`RelationIds`** on the geometries for you.

The two marking `LineString`s use a **±20 px** skew from the parent diagonal endpoints and the same **`marking_path_offset_px`** on both children so the paths stay parallel to the guide in pixel space.

In [ ]:
import uuid

from femtum_sdk.api.contracts.v1.dto.geometry_pb2 import GeometryRelation, LineString

tag = uuid.uuid4().hex[:8]

parent_id = str(uuid.uuid4())
mark_a_id = str(uuid.uuid4())
mark_b_id = str(uuid.uuid4())

p0 = Point(X=cx, Y=cy)
p1 = Point(X=cx + 100, Y=cy + 100)

mark_a_p0 = Point(X=cx - 20, Y=cy + 20)
mark_a_p1 = Point(X=cx + 100 - 20, Y=cy + 100 + 20)

mark_b_p0 = Point(X=cx + 20, Y=cy - 20)
mark_b_p1 = Point(X=cx + 100 + 20, Y=cy + 100 - 20)

offset_a = "46.74107142857143"
offset_b = "46.74107142857143"

Recipe-level **laser** parameters: frequencies plus jump/mark speeds and delays (typical of recipes edited in the GUI).

In [ ]:
laser_params_full = [
    LaserParameter(Key="pulse_train_laser_triggering_frequency_khz", Value="2730", Unit="kHz"),
    LaserParameter(Key="single_pulse_laser_triggering_frequency_khz", Value="2730", Unit="kHz"),
    LaserParameter(Key="laser_jump_speed_mms", Value="1", Unit="mm/s"),
    LaserParameter(Key="laser_mark_speed_mms", Value="1", Unit="mm/s"),
    LaserParameter(Key="delay_mark_us", Value="51", Unit="µs"),
    LaserParameter(Key="delay_jump_us", Value="1800", Unit="µs"),
    LaserParameter(Key="delay_laser_on_us", Value="450", Unit="µs"),
    LaserParameter(Key="delay_laser_off_us", Value="500", Unit="µs"),
]

In [ ]:
offset_recipe = datastore_api.recipe.Create(
    CreateRecipeRequest(
        Name=f"Notebook [wg+2offset sp+pt] {tag}",
        Geometries=[
            Geometry(
                Id=parent_id,
                Type="waveguide",
                Attributes=[Attribute(Key="marking_type", Value="single_pulse")],
                Shape=Shape(LineString=LineString(Points=[p0, p1])),
            ),
            Geometry(
                Id=mark_a_id,
                Type="marking_path",
                Attributes=[
                    Attribute(Key="parent_geometry_id", Value=parent_id),
                    Attribute(Key="marking_path_offset_px", Value=offset_a),
                    Attribute(Key="marking_type", Value="single_pulse"),
                ],
                Shape=Shape(LineString=LineString(Points=[mark_a_p0, mark_a_p1])),
            ),
            Geometry(
                Id=mark_b_id,
                Type="marking_path",
                Attributes=[
                    Attribute(Key="parent_geometry_id", Value=parent_id),
                    Attribute(Key="marking_path_offset_px", Value=offset_b),
                    Attribute(Key="marking_type", Value="pulse_train"),
                ],
                Shape=Shape(LineString=LineString(Points=[mark_b_p0, mark_b_p1])),
            ),
        ],
        Relations=[
            GeometryRelation(
                Type="offset",
                GeometryIds=[parent_id, mark_a_id],
                Attributes=[
                    Attribute(Key="distance_px", Value="50"),
                    Attribute(Key="marking_path_offset_px", Value=offset_a),
                    Attribute(Key="marking_type", Value="single_pulse"),
                ],
            ),
            GeometryRelation(
                Type="offset",
                GeometryIds=[parent_id, mark_b_id],
                Attributes=[
                    Attribute(Key="distance_px", Value="-50"),
                    Attribute(Key="marking_path_offset_px", Value=offset_b),
                    Attribute(Key="marking_type", Value="pulse_train"),
                ],
            ),
        ],
        Attributes=[
            Attribute(Key="context.scaling_factor_mm_per_px", Value=SCALING_MM_PER_PX),
            Attribute(Key="context.width", Value=str(CONTEXT_WIDTH)),
            Attribute(Key="context.height", Value=str(CONTEXT_HEIGHT)),
        ],
        LaserParameters=laser_params_full,
    )
)
print(f"Created id={offset_recipe.Id!r} name={offset_recipe.Name!r}")

## Optional — Refetch as JSON

Use **`FindById`** to load what the server stored. If you omitted relation ids, **`Relations[].Id`** will be server-generated; **`Geometries[].RelationIds`** is filled automatically.

In [ ]:
import json

from google.protobuf.json_format import MessageToDict
from femtum_sdk.api.contracts.v1.storage.recipe_pb2 import FindRecipeByIdRequest

loaded = datastore_api.recipe.FindById(FindRecipeByIdRequest(Id=offset_recipe.Id)).Recipe
print(json.dumps(MessageToDict(loaded, preserving_proto_field_name=True), indent=2))

## Cleanup

Close the datastore client when you are done. The optional hardware block above uses a **`with`** statement, so that client is already closed after reading the camera.

In [ ]:
datastore_api.__exit__(None, None, None)